# Phase 4: Regional Results by Scenario & Protected Area Analysis

## Objective
Identify which 1km² grid cells fall within **Protected Areas (PAs)**. We will then link these spatial locations with the **Natural Regrowth (NR)** mitigation potential calculated in the data analysis phase to quantify the specific contribution of PAs to climate mitigation.

## Workflow
1.  **Spatial Identification:** Use a **Spatial Join** between the 1km Grid Centroids and the Protected Areas shapefile to flag cells located in conservation zones.
2.  **Data Integration:** Import the `NR_Scenario.xlsx` results.
3.  **Join:** Link the NR potential values to the spatial features using the unique ID (`GRI_1K_F_1`).
4.  **Calculation:** Sum the mitigation potential specifically for the Protected Area subset.

## 4.1. Coordinate Reference System (CRS) Alignment

**Objective:** Before performing spatial joins, it is critical to verify that both the **1km² Grid** and the **Protected Areas** shapefile share the same Coordinate Reference System (CRS). 

**Rationale:** Performing spatial operations on mismatched systems can lead to "ghost" intersections or significant displacement errors, incorrectly allocating coca cultivation to protected zones. If a mismatch is detected, this module will project the Protected Areas layer to match the Grid's CRS.

In [4]:
import arcpy
import os

# Set Workspace and Environment
arcpy.env.workspace = r"M:\Documents\Spatial_Analysis_Project\data\interim\Spatial_Analysis\Spatial_Analysis.gdb"
arcpy.env.overwriteOutput = True

# Define Inputs
grid_fc = r"M:\Documents\Spatial_Analysis_Project\data\raw\Grillas1k_Coca_2022\Grillas1k.shp"
pa_fc = r"M:\Documents\Spatial_Analysis_Project\data\interim\Biodiversity_Protected_Areas_ExportFeatures.shp"

print("--- Step 3.1: Validating Coordinate Systems ---")

def validate_and_project(source, reference):
    sr_source = arcpy.Describe(source).spatialReference
    sr_ref = arcpy.Describe(reference).spatialReference
    
    print(f"Layer: {os.path.basename(source)} | CRS: {sr_source.name}")
    print(f"Layer: {os.path.basename(reference)} | CRS: {sr_ref.name}")
    
    if sr_source.name != sr_ref.name:
        print(f"⚠️ Mismatch detected. Projecting {os.path.basename(source)} to match {os.path.basename(reference)}. Please check CRS of projected PAs shapefile...")
        projected_output = f"{os.path.basename(source).split('.')[0]}_Projected"
        arcpy.management.Project(source, projected_output, sr_ref)
        return projected_output
    else:
        print("✅ CRS match confirmed.")
        return source

# Run validation and get the safe PA layer to use
final_pa_fc = validate_and_project(pa_fc, grid_fc)

--- Step 3.1: Validating Coordinate Systems ---
Layer: Biodiversity_Protected_Areas_ExportFeatures.shp | CRS: GCS_WGS_1984
Layer: Grillas1k.shp | CRS: WGS_1984_UTM_Zone_18N
⚠️ Mismatch detected. Projecting Biodiversity_Protected_Areas_ExportFeatures.shp to match Grillas1k.shp...


## 4.2. Comparative Spatial Analysis: Polygons vs. Centroids

**Objective:** Identify the specific grid cells located within Protected Areas using two different geometric methods to evaluate sensitivity.

**Methodology:** 1. **Method A (Polygon Intersection):** Selects any 1km² cell that has *any* overlap with a Protected Area. This is a "liberal" approach that includes boundary-spanning cells.
2. **Method B (Centroid Intersection):** Selects only cells whose geometric center (Point) falls within a Protected Area. This is a "conservative" approach that avoids double-counting and ensures the majority of the cell is protected.

In [7]:
print("\n--- Step 3.2: Comparative Spatial Analysis (Count & Area) ---")

# --- Method A: 1km² Grid Polygons (INTERSECT) ---
pa_grid_joined = "Grid_SpatialJoin_PA"
arcpy.analysis.SpatialJoin(
    target_features=grid_fc,
    join_features=final_pa_fc,
    out_feature_class=pa_grid_joined,
    join_operation="JOIN_ONE_TO_ONE",
    join_type="KEEP_COMMON",
    match_option="INTERSECT"
)

# --- Method B: Grid Centroids (POINT IN POLYGON) ---
# Generate Centroids
grid_centroids = "Grid_Centroids_Temp"
arcpy.management.FeatureToPoint(grid_fc, grid_centroids, "INSIDE")

pa_centroid_joined = "Centroids_SpatialJoin_PA"
arcpy.analysis.SpatialJoin(
    target_features=grid_centroids,
    join_features=final_pa_fc,
    out_feature_class=pa_centroid_joined,
    join_operation="JOIN_ONE_TO_ONE",
    join_type="KEEP_COMMON",
    match_option="INTERSECT"
)

# --- AREA CALCULATION LOGIC ---

def get_total_coca_area(feature_class):
    """Sums the Coca_2022 field for a given feature class"""
    total_area = 0
    # Use SearchCursor to sum the area field
    with arcpy.da.SearchCursor(feature_class, ["Coca_2022"]) as cursor:
        for row in cursor:
            if row[0] is not None:
                total_area += row[0]
    return total_area

# Calculate Counts
count_grid = int(arcpy.management.GetCount(pa_grid_joined)[0])
count_centroid = int(arcpy.management.GetCount(pa_centroid_joined)[0])

# Calculate Areas (Sum of Coca_2022)
area_grid = get_total_coca_area(pa_grid_joined)
area_centroid = get_total_coca_area(pa_centroid_joined)

# --- Final Reporting ---
print("-" * 60)
print(f"{'Metric':<30} | {'Method A (Intersect)':<20} | {'Method B (Centroid)':<20}")
print("-" * 60)
print(f"{'Cell Count':<30} | {count_grid:<20} | {count_centroid:<20}")
print(f"{'Total Coca Area (Ha)':<30} | {area_grid:<20.2f} | {area_centroid:<20.2f}")
print("-" * 60)
print(f"Difference in Territory: {abs(area_grid - area_centroid):.2f} Ha")
print(f"Percentage of Overlap cells: {((count_grid - count_centroid) / count_grid * 100):.2f}%")
print("-" * 60)


--- Step 3.2: Comparative Spatial Analysis (Count & Area) ---
------------------------------------------------------------
Metric                         | Method A (Intersect) | Method B (Centroid) 
------------------------------------------------------------
Cell Count                     | 7709                 | 6217                
Total Coca Area (Ha)           | 32866.95             | 24598.09            
------------------------------------------------------------
Difference in Territory: 8268.86 Ha
Percentage of Overlap cells: 19.35%
------------------------------------------------------------


## 3.3. Integration of Spatial Subsets with NR Mitigation Data

**Objective:** To quantify the climate mitigation potential ($Mg CO_2e$) specifically for Protected Areas.

**Workflow:** 1. **Import Results:** Load the `NR_Scenario.xlsx` file produced during the Data Analysis phase.
2. **Tabular Join:** Link the statistical results to the spatial centroid subset using the unique ID `GRI_1K_F_1`.
3. **Summary Statistics:** Aggregate the total potential and uncertainty (STD) for all cells within the Protected Areas.

**Note:** In these areas, we only consider Natural Regrowth (NR) as the viable mitigation strategy, as biodiversity laws generally prohibit crop-based AFS or bioenergy systems.

In [8]:
print("\n--- Step 3.3: Integrating NR Scenario Data ---")

# Define Paths to Data Analysis Results
nr_excel = r"M:\Documents\Spatial_Analysis_Project\data\results\NR_Scenario.xlsx"
nr_table = "NR_Results_Table"

# Convert Excel to Geodatabase Table
arcpy.conversion.ExcelToTable(nr_excel, nr_table)

# Join NR Potential and STD to the Centroid Join Result
join_field = "GRI_1K_F_1"
fields_to_add = ["NR_Total_Potential_MgCO2_yr", "NR_Intracell_STD_MgCO2_yr", "NR_Error_Ratio_MgCO2_yr"]

arcpy.management.JoinField(
    in_data=pa_centroid_joined,
    in_field=join_field,
    join_table=nr_table,
    join_field=join_field,
    fields=fields_to_add
)

# Final Summary Calculation for the Entire Protected Area Network
out_stats_table = "NR_PA_Summary_Stats"
arcpy.analysis.Statistics(
    in_table=pa_centroid_joined,
    out_table=out_stats_table,
    statistics_fields=[
        ["NR_Total_Potential_MgCO2_yr", "SUM"],
        ["NR_Intracell_STD_MgCO2_yr", "SUM"],
        ["NR_Error_Ratio_MgCO2_yr", "SUM"]
    ]
)

print(f"✅ PA Analysis complete. Summary table created: {out_stats_table}")


--- Step 3.3: Integrating NR Scenario Data ---
✅ PA Analysis complete. Summary table created: NR_PA_Summary_Stats


## 3.4. Master Data Integration: Consolidation of Mitigation Scenarios

**Objective:** Before performing regional clipping, this module consolidates all scenario outputs (Natural Regrowth, Agroforestry, Bioenergy, and BECCS) into a single master centroid feature class.

**Workflow:**
1. **Centroid Creation:** Generate a master point layer from the original 1km² Grid.
2. **Tabular Joining:** Sequentially link the four Excel result files produced in the Data Analysis phase.
3. **Data Integrity:** Ensure the `GRI_1K_F_1` field is used as the primary key to maintain spatial consistency across all scenarios.

In [11]:
import arcpy

# Setup paths
results_folder = r"M:\Documents\Spatial_Analysis_Project\data\results"
master_centroids = "Master_Coca_Centroids_Consolidated"

# 1. Create the base centroids from the 1km Grid
print("Generating Master Centroids...")
arcpy.management.FeatureToPoint(grid_fc, master_centroids, "INSIDE")

# 2. List of Scenario Files and their relevant columns. Note: import Life cycle emissions contributions if you want to calculate those regionally after
scenarios = {
    "NR_Scenario.xlsx": ["NR_Total_Potential_MgCO2_yr", "NR_Error_Ratio_MgCO2_yr", "NR_Intracell_STD_MgCO2_yr"],
    "AFS_Scenario.xlsx": ["AFS_Total_Net_Potential_MgCO2_yr", "AFS_Intracell_STD_MgCO2_yr"],
    "BE_Scenario.xlsx": ["Avoided_Emissions_BE_MgCO2", "Avoided_Emissions_BE_Intracell_STD_MgCO2"],
    "BECCS_Scenario.xlsx": ["Avoided_Emissions_BECCS_MgCO2", "Avoided_Emissions_BECCS_Intracell_STD_MgCO2"]
}

print("Integrating scenario data into centroids...")
for excel_name, fields in scenarios.items():
    excel_path = os.path.join(results_folder, excel_name)
    temp_table = f"Table_{excel_name.split('.')[0]}"
    
    # Convert Excel to Table and Join
    arcpy.conversion.ExcelToTable(excel_path, temp_table)
    arcpy.management.JoinField(master_centroids, "GRI_1K_F_1", temp_table, "GRI_1K_F_1", fields)
    print(f"   ✅ Joined: {excel_name}")

print("Master Spatial Layer Ready.")

Generating Master Centroids...
Integrating scenario data into centroids...
   ✅ Joined: NR_Scenario.xlsx
   ✅ Joined: AFS_Scenario.xlsx
   ✅ Joined: BE_Scenario.xlsx
   ✅ Joined: BECCS_Scenario.xlsx
Master Spatial Layer Ready.


## 3.5. Regional Aggregation by Administrative Department

**Objective:** Perform a spatial join between the consolidated master centroids and the Regional boundaries (`Regions.shp`) to calculate mitigation potential by Department (`DPTO_CNMBR`).

**Methodology:**
* **Spatial Join:** Points are assigned the attributes of the region they fall within.
* **Summary Statistics:** Aggregate the total $CO_2e$ and Uncertainty for all four scenarios, grouped by the Department name.
* **Conservation Alignment:** This allows for a direct comparison of which regions offer the highest potential for specific land-use transitions.

In [15]:
import arcpy
import os

# --- Setup Paths ---
master_centroids = "Master_Centroids_by_Region" # Use the output from the regional join
output_folder = r"M:\Documents\Spatial_Analysis_Project\data\results\Regional_Exports"

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# Your specific region names
target_regions = ["Pacific", "Putumayo-Caqueta", "Catatumbo", "Central", "Eastern Savannas"]

print(f"--- Step 3.6: Exporting Regional Scenarios ---")

try:
    # 1. Identify the correct field name dynamically
    fields = [f.name for f in arcpy.ListFields(master_centroids)]
    region_field = None
    
    # Priority search for the field name
    for f in fields:
        if f.upper() in ["DPTO_CNMBR", "REGION", "REGION_NAME", "DPTO_CNMBR_1"]:
            region_field = f
            break
            
    if not region_field:
        raise ValueError(f"Could not find a region field in {fields}. Please check the attribute table.")

    print(f"Using field '{region_field}' for regional filtering.")

    # 2. Iterate and Export
    for region in target_regions:
        safe_name = region.replace(" ", "_").replace("-", "_")
        out_shp = os.path.join(output_folder, f"Coca_Mitigation_{safe_name}.shp")
        
        # SQL Query with dynamic field name
        where_clause = f"{region_field} = '{region}'"
        
        # Create Feature Layer for selection
        temp_lyr = "selection_lyr"
        if arcpy.Exists(temp_lyr): arcpy.management.Delete(temp_lyr)
        
        arcpy.management.MakeFeatureLayer(master_centroids, temp_lyr)
        
        try:
            arcpy.management.SelectLayerByAttribute(temp_lyr, "NEW_SELECTION", where_clause)
            
            count = int(arcpy.management.GetCount(temp_lyr)[0])
            if count > 0:
                arcpy.management.CopyFeatures(temp_lyr, out_shp)
                print(f"   ✅ Success: {region} ({count} cells) -> {os.path.basename(out_shp)}")
            else:
                print(f"   ⚠️ No features found for {region}. Check for spelling/case mismatches.")
        
        except Exception as select_error:
            print(f"   ❌ Failed to select/export {region}: {select_error}")
        
        finally:
            arcpy.management.Delete(temp_lyr)

    print("\nRegional Spatial Analysis Complete.")

except Exception as e:
    print(f"❌ Critical Error: {e}")

--- Step 3.6: Exporting Regional Scenarios ---
Using field 'DPTO_CNMBR' for regional filtering.
   ✅ Success: Pacific (15000 cells) -> Coca_Mitigation_Pacific.shp
   ✅ Success: Putumayo-Caqueta (8956 cells) -> Coca_Mitigation_Putumayo_Caqueta.shp
   ✅ Success: Catatumbo (5284 cells) -> Coca_Mitigation_Catatumbo.shp
   ✅ Success: Central (9743 cells) -> Coca_Mitigation_Central.shp
   ✅ Success: Eastern Savannas (3100 cells) -> Coca_Mitigation_Eastern_Savannas.shp

Regional Spatial Analysis Complete.


## 3.6. Regional Spatial Exports & Scenario Filtering

**Objective:** To generate independent regional datasets where the attribute table contains the full multi-scenario results. 

**Data Engineering Potential:**
With these shapefiles, we can now filter by attributes to determine:
1.  **Conservation Contribution:** By selecting grid cells that intersect the **Protected Areas (PAs)** file, we can isolate the contribution of Natural Regrowth (NR) vs. other scenarios in restricted zones.
2.  **Scenario Dominance:** Calculate which scenario provides the maximum mitigation per specific regional grid cell.

**Final Step:** This concludes the development of the NR, AFS, BE, and BECCS spatial integration. Use Data engineering tool for each region or national level shapefile, select by attributes to filter the desired outcomes